In [9]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
import matplotlib as mpl
# from colorspacious import cspace_converter
from pathlib import Path
import glob
import os
# import sklearn.metrics as skm
# import cv2
from pathml.core import HESlide
data_path=Path('/home/brian/data/biliseq_proc')
# model_path=Path('/home/brian/data/data/biliseq_proc/v6')
model_path=Path('/home/brian/odrive/project_data/ASinghi-068/data/biliseq_he_class/proc/v6/')
svs_path=Path('/home/brian/data/biliseq_proc/v6/svs')
tiles_path=Path('/home/brian/data/biliseq_proc/v6/tiles')
print('Finished')
svs_path.exists()

Finished


True

In [7]:
class MplColorHelper:
  def __init__(self, cmap_name, start_val, stop_val):
    self.cmap_name = cmap_name
    self.cmap = plt.get_cmap(cmap_name)
    self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
    self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

  def get_rgb(self, val):
    return self.scalarMap.to_rgba(val)

COL = MplColorHelper('RdYlBu', 0, 1)
im = np.zeros((50,50,3))
rgb = COL.get_rgb(0.5)
print(rgb)

(0.9976163014225298, 0.9990772779700116, 0.7534025374855825, 1.0)


In [8]:
def parse_x_y_from_fn(fn):
    fn=Path(fn)
    x=int(fn.parts[-1].split('_')[3][1:])
    y=int(fn.parts[-1].split('_')[4].split('.')[0][1:])
    return x,y
cmap = 'RdYlBu'
# cmap = 'OrRd'
COL = MplColorHelper(cmap, 0, 1)
fold = 4
use='tiles'
model_data='%s_model' % use
model_type='resnet18_jackknife_v0'
total_slides=35
infer_path=model_path.joinpath(model_data).joinpath(model_type).joinpath('infer_csv')
df=pd.read_csv(infer_path.joinpath('fold_%d_all_valid_pred.csv' % fold))
base_file=df.loc[0,'slide'] #positive slide to examine
wsi = HESlide(str(svs_path.joinpath('%s.svs' % base_file)))
x,y= wsi.shape
print('original slide shape:',x,y)
ds = 20
xds= x//ds
yds= y//ds
print("ds shape:",xds,yds)
blank_image = np.zeros((xds,yds,3), np.uint8) +255
tile_fns = df.loc[:,'fn']
tile_size = 500
ds_ts = tile_size//ds


for i,fn in enumerate(tile_fns):
    tx,ty=parse_x_y_from_fn(fn)
    dsx=tx//ds
    dsy=ty//ds
    im = np.zeros((50,50,3)).astype(np.uint8)
    p = df.loc[i,'p_pos']
    
    rgb = np.array(COL.get_rgb(1-p)[0:-1]) * 255
    rgb = rgb.reshape((1,1,3))
    im = im + rgb
    # im = cv2.imread(str(tiles_path.joinpath(fn)))
    # tile_size=im.shape[0]
    imds = cv2.resize(im,(tile_size//ds,tile_size//ds),interpolation=cv2.INTER_CUBIC)
    # imds = cv2.cvtColor(imds, cv2.COLOR_RGB2BGR)

    blank_image[dsx:(dsx + imds.shape[1]),dsy:(dsy+imds.shape[0])]=imds

fig = plt.figure(figsize=(8,3))
ax = fig.add_subplot(1,1,1)
ax.imshow(blank_image)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/home/brian/data/data/biliseq_proc/v6/tiles_model/resnet18_jackknife_v0/infer_csv/fold_4_all_valid_pred.csv'

In [100]:
imd = cv2.cvtColor(blank_image,cv2.COLOR_RGB2BGR)
cv2.imwrite('/home/brian/data/biliseq_proc/v6/cm_%s.jpg' %base_file,imd)

True

In [ ]:
#Compare low p(pos) tiles vs. high p(pos)
f,ax = plt.subplots(nrows=5,ncols=2))
p_thresh= [0.4, 0.7]
for i,t in enumerate(p_thresh):
    tiles_path

In [21]:
use='tiles'
model_data='%s_model' % use
model_type='resnet18_jackknife_v0'
if '10k' in model_type:
    folds=[10,10]
    nslides=[35,34] #Summary models drop one slide due to insufficient data
elif 'jackknife' in model_type:
    nslides=[35,34]
    folds=[35,34]
total_slides=35
use='tiles'
infer_path=model_path.joinpath(model_data).joinpath(model_type).joinpath('infer_csv')

true_pos=[]
pred_pos=[]
pred_p=[]
slide=[]
thresh=0.7
for fold in range(0,total_slides):
    df=pd.read_csv(infer_path.joinpath('fold_%d_all_valid_pred.csv' % fold))
    true_pos.append(df.loc[0,'class']=='pos')
    m_p=np.mean(df.loc[:,'p_pos'])
    pred_p.append(m_p)
    slide.append(df.loc[0,'slide'])
    pred_pos.append(m_p > thresh)
summary=pd.DataFrame(list(zip(slide,true_pos,pred_pos,pred_p)),columns=['slide','true_pos','pred_pos','p_pos'])
summary
    

,slide,true_pos,pred_pos,p_pos
0,1007466,False,False,0.582492
1,1007467,False,False,0.380821
2,1007468,False,False,0.360764
3,1007469,False,False,0.304951
4,1007470,False,True,0.798407
5,1007471,False,False,0.109159
6,1007473,False,True,0.738982
7,1007474,False,False,0.387552
8,1007476,True,False,0.467094
9,1007477,True,False,0.656944


In [ ]:
ds = 20

xds= x//ds
yds= y//ds
print("ds shape:",xds,yds)
blank_image = np.zeros((xds,yds,3), np.uint8) +255
tile_size
tile_fns=df.loc[:,'fn']
for fn in tile_fns:
    tx,ty=parse_x_y_from_fn(fn)
    dsx=tx//ds
    dsy=ty//ds
    im = cv2.imread(fn)
    tile_size=im.shape[0]
    imds=cv2.resize(im,(tile_size//ds,tile_size//ds),interpolation=cv2.INTER_CUBIC)
    blank_image[dsx:(dsx + imds.shape[1]),dsy:(dsy+imds.shape[0])]=imds

plt.figure()
plt.imshow(blank_image)
dest
cv2.imwrite('/home/brian/biliseq/Bile Ducts - Positive/proc/full_ds/%s.jpg' %base_file,blank_image)

In [ ]:
#Generate fold_summary.csv
nslides=len(np.unique(slides_c))
model_type='resnet18_jackknife_v0'
start=0
stop=nslides
u_slides_c=np.unique(slides_c)
temp = [item.split('_') for item in u_slides_c]
fold_df=pd.DataFrame(temp,columns=['slide','class'])
s=time.time()
model_path='%s_model' % use
export_path=path.joinpath(model_path).joinpath(model_type).joinpath('fold_models')
csv_path=path.joinpath(model_path).joinpath(model_type).joinpath('csv')
infer_path=path.joinpath(model_path).joinpath(model_type).joinpath('infer_csv')

for fold in range(start,stop):
    print('Perform inference on fold :', fold)
    df=pd.read_csv(infer_path.joinpath('fold_%d_all_valid_pred.csv' % fold))
    test_df=df.loc[df.loc[:,'is_valid']==1,:].reset_index(drop=True)
    test_slides=np.unique(test_df.loc[:,'slide'])
    p=np.array(test_df.loc[:,'p_pos']) # Probability ['neg','pos'] (Can check with dls.vocab )
    c=np.array(test_df.loc[:,'pred_cls']) #Predictions decoded
    fold_df['p1_fold_%d' % fold] = np.zeros((fold_df.shape[0],1))*np.nan
    fold_df['wta_fold_%d' % fold] = np.zeros((fold_df.shape[0],1))*np.nan
    for slide in test_slides:
        src_idx = np.array(test_df.loc[:,'slide']) == slide
        p1 = np.mean(p[src_idx])
        if np.sum(c[src_idx]==0) > np.sum(c[src_idx]==1):
            wta=0
        else:
            wta=1
        dest_idx = np.array(fold_df.loc[:,'slide'])== str(slide)
        # print(dest_idx,p1)
        fold_df.loc[dest_idx,'p1_fold_%d' % fold] = p1
        fold_df.loc[dest_idx,'wta_fold_%d' % fold] = wta
ss=time.time()
print('Wall time: %ds' % (ss-s))
# fold_df.head()

fn=infer_path.joinpath('fold_summary.csv')
print('Saving inference summary to %s' % fn)
fold_df.to_csv(fn)
print('Finished')

In [ ]:
weights=[.7,.3]
ans=['tiles','summary']
t=np.array(ens.loc[:,'%s_p1' % 'tiles'])
temp=np.zeros(t.shape)
for w,use in zip(weights,ans):
    temp=np.nansum((temp, np.array(ens.loc[:,'%s_p1' % use])*w),axis=0)
    # temp[np.isnan(temp)]=0.5

m_p1 = temp
# m_p1= np.nanmean((np.array(ens0.loc[:,'tiles_p1']),
#                   np.array(ens2.loc[:,'tiles_p1'])),axis=0)
                  #temp #np.nanmean(ens.loc[:,('tiles_p1','summary_p1')],axis=1)
true_pos = np.array(ens.loc[:,'class'])=='pos'

pred_pos = m_p1 > 0.5
use = ~np.isnan(m_p1)

print(np.sum(pred_pos[use] == true_pos[use]) / np.sum(use) )

fpr,tpr,thresh =skm.roc_curve(true_pos[use],m_p1[use])
pred_pos = m_p1 > 0.5
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1],'--r')
c=skm.confusion_matrix(true_pos[use],pred_pos[use])

auc=skm.auc(fpr,tpr)

plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
print(c)
tn=c[0][0]
tp=c[1][1]
fn=c[1][0]
fp=c[0][1]

sensitivity = tp/(tp + fn) #True pos / all positive
# print('sensitivity',sensitivity)

specificity = tn/(fp + tn)
# print('specificity',specificity) #True neg / all negative
plt.title('AUROC %1.3f, Sensitivity: %1.2f, Specificity: %1.2f'% (auc,sensitivity,specificity))
    